In [0]:
asset_raw_df = (
    spark.read
        .text("s3://enterprise-lakehouse-data/bronze/asset_snapshots/")
)

asset_raw_df.show(5, truncate=False)
asset_raw_df.count()


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|value                                                                                                                                                                                                                                                                                                                             |ingestion_date|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

20100

PARSE

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
payload_schema = StructType([
    StructField("asset_id", StringType(), True),
    StructField("plant_id", StringType(), True),
    StructField("asset_type", StringType(), True),
    StructField("install_date", StringType(), True),
    StructField("capacity_mw", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("last_updated", StringType(), True)
])

envelope_schema = StructType([
    StructField("payload", payload_schema, True),
    StructField("kafka_topic", StringType(), True),
    StructField("kafka_partition", IntegerType(), True),
    StructField("kafka_offset", IntegerType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])


In [0]:
parsed_df = (
    asset_raw_df
        .withColumn("json", from_json(col("value"), envelope_schema))
        .select(
            "json.payload.*",
            "json.ingestion_timestamp",
            "ingestion_date"
        )
)


TYPE CASTING

In [0]:
from pyspark.sql.functions import to_date, to_timestamp
typed_df = (
    parsed_df
        .withColumn("install_date_dt", to_date("install_date"))
        .withColumn("last_updated_dt", to_date("last_updated"))
        .withColumn("ingestion_ts", to_timestamp("ingestion_timestamp"))
)


In [0]:
typed_df.select(
    "asset_id", "asset_type", "capacity_mw",
    "install_date_dt", "last_updated_dt", "ingestion_ts"
).show(10, truncate=False)


+--------+----------+-----------+---------------+---------------+--------------------------+
|asset_id|asset_type|capacity_mw|install_date_dt|last_updated_dt|ingestion_ts              |
+--------+----------+-----------+---------------+---------------+--------------------------+
|AST-1158|PUMP      |200.0      |2025-10-28     |2025-06-17     |2026-01-07 13:49:09.842799|
|AST-1174|PUMP      |50.0       |2025-08-24     |2025-12-17     |2026-01-07 13:49:09.842823|
|AST-1200|PUMP      |-50.0      |2025-08-13     |2026-02-07     |2026-01-07 13:49:09.842826|
|AST-1021|TURBINE   |50.0       |2025-08-31     |2025-06-01     |2026-01-07 13:49:09.842828|
|AST-1029|PUMP      |100.0      |2025-04-12     |2026-07-08     |2026-01-07 13:49:09.842831|
|AST-1129|SOLAR     |100.0      |2025-10-29     |2026-05-07     |2026-01-07 13:49:09.842833|
|AST-1035|SOLAR     |200.0      |2026-04-18     |2025-09-20     |2026-01-07 13:49:09.842835|
|AST-1032|GENERATOR |-100.0     |2026-06-06     |2025-09-20     |2026-

In [0]:
from pyspark.sql.functions import col, length
typed_df.select(
    col("asset_id"),
    col("asset_id").isNull().alias("is_null"),
    (col("asset_id") == "NULL").alias("is_string_NULL"),
    (col("asset_id") == "null").alias("is_string_null"),
    (length(col("asset_id")) == 0).alias("is_empty_string")
).show(20, truncate=False)


+--------+-------+--------------+--------------+---------------+
|asset_id|is_null|is_string_NULL|is_string_null|is_empty_string|
+--------+-------+--------------+--------------+---------------+
|AST-1158|false  |false         |false         |false          |
|AST-1174|false  |false         |false         |false          |
|AST-1200|false  |false         |false         |false          |
|AST-1021|false  |false         |false         |false          |
|AST-1029|false  |false         |false         |false          |
|AST-1129|false  |false         |false         |false          |
|AST-1035|false  |false         |false         |false          |
|AST-1032|false  |false         |false         |false          |
|AST-1071|false  |false         |false         |false          |
|AST-1096|false  |false         |false         |false          |
|AST-1088|false  |false         |false         |false          |
|AST-1035|false  |false         |false         |false          |
|AST-1049|false  |false  

NULL NORMALIZATION

In [0]:
from pyspark.sql.functions import when, trim
normalized_df = (
    typed_df
        .withColumn(
            "asset_id",
            when(
                trim(col("asset_id")).isin("NULL", "null", ""),
                None
            ).otherwise(col("asset_id"))
        )
        .withColumn(
            "asset_type",
            when(
                trim(col("asset_type")).isin("NULL", "null", ""),
                None
            ).otherwise(col("asset_type"))
        )
        .withColumn(
            "status",
            when(
                trim(col("status")).isin("NULL", "null", ""),
                None
            ).otherwise(col("status"))
        )
)


VALIDATION

In [0]:
from pyspark.sql.functions import col
hard_flags_df = (
    typed_df
        .withColumn("f_null_asset_id", col("asset_id").isNull())
        .withColumn("f_null_asset_type", col("asset_type").isNull())
        .withColumn("f_null_status", col("status").isNull())
        .withColumn("f_invalid_capacity", col("capacity_mw") <= 0)
        .withColumn("f_null_ingestion_ts", col("ingestion_ts").isNull())
)
hard_flags_df = hard_flags_df.withColumn(
    "f_invalid_capacity",
    (col("status") == "ACTIVE") & (col("capacity_mw") <= 0)
)


In [0]:
with_failure_df = (
    hard_flags_df
        .withColumn(
            "failure_reason",
            expr("""
                filter(
                    array(
                        CASE WHEN f_null_asset_id THEN 'NULL_ASSET_ID' END,
                        CASE WHEN f_null_asset_type THEN 'NULL_ASSET_TYPE' END,
                        CASE WHEN f_null_status THEN 'NULL_STATUS' END,
                        CASE WHEN f_invalid_capacity THEN 'NEGATIVE_OR_ZERO_CAPACITY_ACTIVE' END,
                        CASE WHEN f_null_ingestion_ts THEN 'NULL_INGESTION_TIMESTAMP' END
                    ),
                    x -> x IS NOT NULL
                )
            """)
        )
)


In [0]:
silver_df = with_failure_df.filter(expr("size(failure_reason) = 0"))
quarantine_df = with_failure_df.filter(expr("size(failure_reason) > 0"))


In [0]:
silver_df.count()

18101

In [0]:
quarantine_df.count()

1999

In [0]:
quarantine_df.select("failure_reason").distinct().show()


+--------------------+
|      failure_reason|
+--------------------+
|[NEGATIVE_OR_ZERO...|
+--------------------+



In [0]:
quarantine_df.groupBy("status").count().show()


+------+-----+
|status|count|
+------+-----+
|ACTIVE| 1999|
+------+-----+



In [0]:
quarantine_df.groupBy("asset_type").count().show()


+----------+-----+
|asset_type|count|
+----------+-----+
|      PUMP|  489|
| GENERATOR|  469|
|   TURBINE|  539|
|     SOLAR|  502|
+----------+-----+



In [0]:
quarantine_df.printSchema()

root
 |-- asset_id: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- asset_type: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- status: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- install_date_dt: date (nullable = true)
 |-- last_updated_dt: date (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- f_null_asset_id: boolean (nullable = false)
 |-- f_null_asset_type: boolean (nullable = false)
 |-- f_null_status: boolean (nullable = false)
 |-- f_invalid_capacity: boolean (nullable = true)
 |-- f_null_ingestion_ts: boolean (nullable = false)
 |-- failure_reason: array (nullable = false)
 |    |-- element: string (containsNull = true)



In [0]:
(
    quarantine_df
        .write
        .mode("overwrite")                 # idempotent per partition
        .partitionBy("ingestion_date")
        .parquet(
            "s3://enterprise-lakehouse-data/quarantine/asset_snapshots/"
        )
)


HASHING

In [0]:
from pyspark.sql.functions import sha2, col, concat_ws
silver_hashed_df = (
    silver_df
        .withColumn(
            "asset_event_hash",
            sha2(
                concat_ws("||", col("asset_id"), col("last_updated_dt").cast("string")),
                256
            )
        )
)



In [0]:
silver_hashed_df.select("asset_id", "asset_event_hash").show(5, truncate=False)


+--------+----------------------------------------------------------------+
|asset_id|asset_event_hash                                                |
+--------+----------------------------------------------------------------+
|AST-1158|ad42d3f6e01c961d82c0f5a85281769c8485f460ced86d5d99c405e6ee639bab|
|AST-1174|c725d83f9e31eab0f79d4f9765f040bb792327bbf04b8f945d1b7cf7fa070a8b|
|AST-1021|6de393a0eaba0aec6d12a349ee784edbe32070da560ed559169e1b175551ab78|
|AST-1029|06de22135d7dfe520a6549822984651eb406858c557074446582295a8007cb80|
|AST-1129|a54e2857e4b3ff0e2dcaee333e6a74033e3b268ab839d0c69de7832f096c0818|
+--------+----------------------------------------------------------------+
only showing top 5 rows


DEDUPE

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
event_dedup_window = (
    Window
        .partitionBy("asset_event_hash")
        .orderBy(col("ingestion_ts").desc())
)
silver_dedup_df = (
    silver_hashed_df
        .withColumn("row_num", row_number().over(event_dedup_window))
        .filter(col("row_num") == 1)
        .drop("row_num")
)


In [0]:
print("Before dedup:", silver_hashed_df.count())
print("After dedup:", silver_dedup_df.count())


Before dedup: 18101
After dedup: 16873


In [0]:
silver_asset_snapshots=silver_dedup_df

In [0]:
(
    silver_asset_snapshots
        .write
        .mode("overwrite")                 # idempotent per partition
        .partitionBy("ingestion_date")
        .parquet(
            "s3://enterprise-lakehouse-data/silver/asset_snapshots/"
        )
)


maintenance_snapshots

In [0]:
maintenance_raw_df = (
    spark.read
        .text("s3://enterprise-lakehouse-data/bronze/maintenance_snapshots/")
)


In [0]:
maintenance_raw_df.show(1, truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|value                                                                                                                                                                                                                                                                                                               |ingestion_date|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|{"payload": {"mainten

In [0]:
maintenance_raw_df.show(1, truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|value                                                                                                                                                                                                                                                                                                               |ingestion_date|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|{"payload": {"mainten

DEFINE SCHEMA

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
maintenance_payload_schema = StructType([
    StructField("maintenance_id", StringType(), True),
    StructField("asset_id", StringType(), True),
    StructField("maintenance_type", StringType(), True),
    StructField("cost", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("actual_date", StringType(), True)
])

maintenance_envelope_schema = StructType([
    StructField("payload", maintenance_payload_schema, True),
    StructField("kafka_topic", StringType(), True),
    StructField("kafka_partition", IntegerType(), True),
    StructField("kafka_offset", IntegerType(), True),
    StructField("ingestion_timestamp", StringType(), True)
])


PARSE JSON

In [0]:
from pyspark.sql.functions import from_json, col
maintenance_parsed_df = (
    maintenance_raw_df
        .withColumn(
            "json",
            from_json(col("value"), maintenance_envelope_schema)
        )
        .select(
            col("json.payload.*"),
            col("json.ingestion_timestamp"),
            col("ingestion_date")
        )
)


In [0]:
maintenance_parsed_df.printSchema()
maintenance_parsed_df.show(5, truncate=False)


root
 |-- maintenance_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- status: string (nullable = true)
 |-- actual_date: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)

+--------------+--------+----------------+-------+---------+-----------+---------------------------+--------------+
|maintenance_id|asset_id|maintenance_type|cost   |status   |actual_date|ingestion_timestamp        |ingestion_date|
+--------------+--------+----------------+-------+---------+-----------+---------------------------+--------------+
|MNT-5083      |AST-1069|BREAKDOWN       |25000.0|CANCELLED|2026-01-07 |2026-01-07T13:49:20.412727Z|2026-01-07    |
|MNT-5279      |AST-1055|PREVENTIVE      |12000.0|COMPLETED|2026-01-07 |2026-01-07T13:49:20.412742Z|2026-01-07    |
|MNT-5498      |AST-1134|BREAKDOWN       |-5000.0|CANCELLED|NULL    

TYPE CASTING

In [0]:
from pyspark.sql.functions import to_date, to_timestamp
maintenance_typed_df = (
    maintenance_parsed_df
        .withColumn("actual_date_dt", to_date("actual_date"))
        .withColumn("ingestion_ts", to_timestamp("ingestion_timestamp"))
)


CHECKPOINT

In [0]:
maintenance_typed_df.select(
    "maintenance_id",
    "asset_id",
    "maintenance_type",
    "cost",
    "status",
    "actual_date_dt",
    "ingestion_ts"
).show(10, truncate=False)


+--------------+--------+----------------+--------+---------+--------------+--------------------------+
|maintenance_id|asset_id|maintenance_type|cost    |status   |actual_date_dt|ingestion_ts              |
+--------------+--------+----------------+--------+---------+--------------+--------------------------+
|MNT-5083      |AST-1069|BREAKDOWN       |25000.0 |CANCELLED|2026-01-07    |2026-01-07 13:49:20.412727|
|MNT-5279      |AST-1055|PREVENTIVE      |12000.0 |COMPLETED|2026-01-07    |2026-01-07 13:49:20.412742|
|MNT-5498      |AST-1134|BREAKDOWN       |-5000.0 |CANCELLED|NULL          |2026-01-07 13:49:20.412746|
|MNT-5209      |AST-1077|PREVENTIVE      |5000.0  |COMPLETED|NULL          |2026-01-07 13:49:20.412748|
|MNT-5007      |AST-1127|BREAKDOWN       |-5000.0 |CANCELLED|2026-01-07    |2026-01-07 13:49:20.41275 |
|MNT-5512      |AST-1143|BREAKDOWN       |5000.0  |CANCELLED|NULL          |2026-01-07 13:49:20.412752|
|MNT-5019      |AST-1117|PREVENTIVE      |12000.0 |CANCELLED|202

VALIDATION FLAGS

In [0]:
from pyspark.sql.functions import col
maintenance_flags_df = (
    maintenance_typed_df
        .withColumn("f_null_maintenance_id", col("maintenance_id").isNull())
        .withColumn("f_null_asset_id", col("asset_id").isNull())
        .withColumn("f_null_maintenance_type", col("maintenance_type").isNull())
        .withColumn("f_null_status", col("status").isNull())
        .withColumn("f_negative_cost", col("cost") < 0)
        .withColumn("f_null_ingestion_ts", col("ingestion_ts").isNull())
)


failure_reason ARRAY

In [0]:
from pyspark.sql.functions import expr
maintenance_with_failure_df = (
    maintenance_flags_df
        .withColumn(
            "failure_reason",
            expr("""
                filter(
                    array(
                        CASE WHEN f_null_maintenance_id THEN 'NULL_MAINTENANCE_ID' END,
                        CASE WHEN f_null_asset_id THEN 'NULL_ASSET_ID' END,
                        CASE WHEN f_null_maintenance_type THEN 'NULL_MAINTENANCE_TYPE' END,
                        CASE WHEN f_null_status THEN 'NULL_STATUS' END,
                        CASE WHEN f_negative_cost THEN 'NEGATIVE_COST' END,
                        CASE WHEN f_null_ingestion_ts THEN 'NULL_INGESTION_TIMESTAMP' END
                    ),
                    x -> x IS NOT NULL
                )
            """)
        )
)


SPLIT SILVER vs QUARANTINE

In [0]:
maintenance_silver_df = maintenance_with_failure_df.filter(expr("size(failure_reason) = 0"))
maintenance_quarantine_df = maintenance_with_failure_df.filter(expr("size(failure_reason) > 0"))


In [0]:
print("Total:", maintenance_typed_df.count())
print("Silver:", maintenance_silver_df.count())
print("Quarantine:", maintenance_quarantine_df.count())


Total: 5200
Silver: 3666
Quarantine: 1534


In [0]:
maintenance_quarantine_df.select(
    "maintenance_id",
    "cost",
    "failure_reason"
).show(20, truncate=False)


+--------------+--------+---------------+
|maintenance_id|cost    |failure_reason |
+--------------+--------+---------------+
|MNT-5498      |-5000.0 |[NEGATIVE_COST]|
|MNT-5007      |-5000.0 |[NEGATIVE_COST]|
|MNT-5427      |-12000.0|[NEGATIVE_COST]|
|MNT-5241      |-12000.0|[NEGATIVE_COST]|
|MNT-5317      |-5000.0 |[NEGATIVE_COST]|
|MNT-5290      |-12000.0|[NEGATIVE_COST]|
|MNT-5219      |-5000.0 |[NEGATIVE_COST]|
|MNT-5204      |-12000.0|[NEGATIVE_COST]|
|MNT-5567      |-12000.0|[NEGATIVE_COST]|
|MNT-5343      |-5000.0 |[NEGATIVE_COST]|
|MNT-5106      |-5000.0 |[NEGATIVE_COST]|
|MNT-5064      |-5000.0 |[NEGATIVE_COST]|
|MNT-5184      |-12000.0|[NEGATIVE_COST]|
|MNT-5415      |-5000.0 |[NEGATIVE_COST]|
|MNT-5448      |-5000.0 |[NEGATIVE_COST]|
|MNT-5392      |-25000.0|[NEGATIVE_COST]|
|MNT-5380      |-25000.0|[NEGATIVE_COST]|
|MNT-5292      |-5000.0 |[NEGATIVE_COST]|
|MNT-5538      |-25000.0|[NEGATIVE_COST]|
|MNT-5530      |-5000.0 |[NEGATIVE_COST]|
+--------------+--------+---------

In [0]:
(
    maintenance_quarantine_df
        .write
        .mode("overwrite")                 # idempotent per partition
        .partitionBy("ingestion_date")
        .parquet(
            "s3://enterprise-lakehouse-data/quarantine/maintenance_snapshots/"
        )
)


HASHING (EVENT-LEVEL, NOT ASSET-LEVEL)

In [0]:
from pyspark.sql.functions import sha2, concat_ws, col
maintenance_hashed_df = (
    maintenance_silver_df
        .withColumn(
            "maintenance_event_hash",
            sha2(
                concat_ws(
                    "||",
                    col("maintenance_id"),
                    col("status"),
                    col("actual_date_dt").cast("string")
                ),
                256
            )
        )
)



In [0]:
maintenance_hashed_df.select("maintenance_event_hash").show()

+----------------------+
|maintenance_event_hash|
+----------------------+
|  9c8565afd23263a3d...|
|  ebfd90d3dc921ab3f...|
|  0f0d51ebbbd953c98...|
|  1abf984090d7fff8f...|
|  01a6bf7c55ff61384...|
|  d8656dedb3bea5728...|
|  097c6daa77d40828a...|
|  4f681a28eb197b104...|
|  fec59edeeb52fa8f5...|
|  c828a4762aaa8bb5a...|
|  aafd9596fb2f4a0b2...|
|  0669eaae3e4395a4e...|
|  0f5d13cb64cc37172...|
|  f6ef85ed4bc0ec84b...|
|  a9c0744fb970644c3...|
|  a0bcc7999f8a86f9f...|
|  af5a2d6ceda17e095...|
|  99557fec5ee996cf6...|
|  b3dcd01ecfd22a865...|
|  19a67414d6ca995f0...|
+----------------------+
only showing top 20 rows


DEDUPLICATION

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
dedup_window = (
    Window
        .partitionBy("maintenance_event_hash")
        .orderBy(col("ingestion_ts").desc())
)

maintenance_dedup_df = (
    maintenance_hashed_df
        .withColumn("row_num", row_number().over(dedup_window))
        .filter(col("row_num") == 1)
        .drop("row_num")
)



In [0]:
print("Before dedup:", maintenance_hashed_df.count())
print("After dedup:", maintenance_dedup_df.count())


Before dedup: 3666
After dedup: 2287


WATERMARKING

In [0]:
from pyspark.sql.functions import datediff, when, lit
maintenance_late_df = (
    maintenance_dedup_df
        .withColumn(
            "lateness_days",
            datediff(col("ingestion_date"), col("actual_date_dt"))
        )
        .withColumn(
            "is_late_record",
            when(col("lateness_days") > 1, lit(True)).otherwise(lit(False))
        )
)


In [0]:
maintenance_silver_final_df = maintenance_late_df.select(
    "maintenance_id",
    "asset_id",
    "maintenance_type",
    "status",
    "cost",
    "actual_date_dt",

    "maintenance_event_hash",
    "ingestion_date",

    "lateness_days",
    "is_late_record"
)


In [0]:
maintenance_silver_final_df.printSchema()

root
 |-- maintenance_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- actual_date_dt: date (nullable = true)
 |-- maintenance_event_hash: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- lateness_days: integer (nullable = true)
 |-- is_late_record: boolean (nullable = false)



In [0]:
(
    maintenance_silver_final_df
        .write
        .mode("overwrite")
        .partitionBy("ingestion_date")
        .parquet("s3://enterprise-lakehouse-data/silver/maintenance_snapshots/")
)


In [0]:
maintenance_dedup_df.groupBy("maintenance_id").count().orderBy("count", ascending=False).show(10)


+--------------+-----+
|maintenance_id|count|
+--------------+-----+
|      MNT-5011|    6|
|      MNT-5595|    6|
|      MNT-5571|    6|
|      MNT-5591|    6|
|      MNT-5474|    6|
|      MNT-5565|    6|
|      MNT-5092|    6|
|      MNT-5199|    6|
|      MNT-5017|    6|
|      MNT-5152|    6|
+--------------+-----+
only showing top 10 rows
